In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
from mlflow import MlflowClient

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [0]:
!pip install lightgbm

In [0]:
# Dataset path
DATA_PATH = "demand_forecasting.csv"

# MLflow experiment
EXPERIMENT_NAME = "/Shared/demand-forecasting"

# Name of the MLflow run we want to evaluate
RUN_NAME = "lightgbm_demand_forecasting"

TARGET = "Demand"

GROUP_COLS = ["Store ID","Product ID"]

In [0]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

display(df.head())

In [0]:
df["Date"] = pd.to_datetime(df["Date"])

df = (df.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True))

print("Date range:")
print(df["Date"].min(), "to", df["Date"].max())

In [0]:
df["day_of_week"] = df["Date"].dt.dayofweek
df["day_of_month"] = df["Date"].dt.day
df["week_of_year"] = df["Date"].dt.isocalendar().week.astype(int)
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter
df["year"] = df["Date"].dt.year
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

display(
    df[
        [
            "Date",
            "day_of_week",
            "day_of_month",
            "week_of_year",
            "month",
            "quarter",
            "year",
            "is_weekend"
        ]
    ].head()
)

In [0]:
LAG_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

for lag in [1, 7, 14, 28]:

    df[f"lag_{lag}"] = (
        df.groupby(GROUP_COLS)["Demand"]
          .shift(lag)
    )

display(
    df[
        GROUP_COLS +
        ["Date", "Demand"] +
        LAG_FEATURES
    ].head(35)
)

In [0]:
ROLLING_MEAN_FEATURES = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

df["rolling_mean_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .mean()
      )
)

df["rolling_mean_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(14)
           .mean()
      )
)

df["rolling_mean_28"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(28)
           .mean()
      )
)

In [0]:
ROLLING_STD_FEATURES = [
    "rolling_std_7",
    "rolling_std_14"
]

df["rolling_std_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .std()
      )
)

df["rolling_std_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(14)
           .std()
      )
)

In [0]:
HISTORY_FEATURES = (
    LAG_FEATURES
    + ROLLING_MEAN_FEATURES
    + ROLLING_STD_FEATURES
)

print("Rows before dropping:", len(df))

df_model = df.dropna(
    subset=HISTORY_FEATURES
).copy()

print("Rows after dropping:", len(df_model))

In [0]:
CATEGORICAL_FEATURES = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
    "Epidemic"
]

BUSINESS_FEATURES = [
    "Inventory Level",
    "Units Ordered",
    "Price",
    "Discount",
    "Promotion",
    "Competitor Pricing"
]

TIME_FEATURES = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend"
]

FEATURES = (
    LAG_FEATURES
    + ROLLING_MEAN_FEATURES
    + ROLLING_STD_FEATURES
    + BUSINESS_FEATURES
    + TIME_FEATURES
    + CATEGORICAL_FEATURES
)

print("Number of features:", len(FEATURES))

print("\nFeatures:")
for feature in FEATURES:
    print(feature)

In [0]:
missing_features = [
    feature
    for feature in FEATURES
    if feature not in df_model.columns
]

if missing_features:
    raise ValueError(
        f"Missing features: {missing_features}"
    )

print("All features are available.")

In [0]:
for col in CATEGORICAL_FEATURES:
    df_model[col] = df_model[col].astype("category")

print("Categorical columns converted.")

In [0]:
dates = sorted(df_model["Date"].unique())

train_end = dates[int(len(dates) * 0.70)]

valid_end = dates[int(len(dates) * 0.85)]

train_df = df_model[df_model["Date"] <= train_end].copy()

valid_df = df_model[(df_model["Date"] > train_end) &(df_model["Date"] <= valid_end)].copy()

test_df = df_model[df_model["Date"] > valid_end].copy()

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

In [0]:
print("TRAIN")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nVALIDATION")
print(valid_df["Date"].min(), "to", valid_df["Date"].max())

print("\nTEST")
print(test_df["Date"].min(), "to", test_df["Date"].max())

In [0]:
X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [0]:
mlflow.set_tracking_uri("databricks")

client = MlflowClient()

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("Experiment ID:", experiment.experiment_id)

In [0]:
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.runName = '{RUN_NAME}'",
    order_by=["start_time DESC"]
)

if not runs:
    raise ValueError(
        f"No MLflow run found with name: {RUN_NAME}"
    )

run = runs[0]

run_id = run.info.run_id

print("Run ID:", run_id)
print("Run name:", run.data.tags.get("mlflow.runName"))

In [0]:
model_uri = f"runs:/{run_id}/model"

print("Model URI:")
print(model_uri)

model = mlflow.lightgbm.load_model(model_uri)

print("Model loaded successfully.")

In [0]:
y_pred = model.predict(X_test)

# Demand cannot normally be negative
y_pred = np.maximum(y_pred, 0)

print("Predictions generated.")

display(
    pd.DataFrame({
        "Actual": y_test.values,
        "Predicted": y_pred
    }).head(20)
)

In [0]:
def safe_mape(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        ) * 100
    )

In [0]:
mae = mean_absolute_error(y_test,y_pred)

rmse = np.sqrt(mean_squared_error(y_test,y_pred))

r2 = r2_score(y_test,y_pred)

mape = safe_mape(y_test,y_pred)

print("LightGBM Test Performance")
print("-------------------------")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

In [0]:
baseline_pred = test_df["lag_1"].values

baseline_pred = np.maximum(
    baseline_pred,
    0
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

baseline_mape = safe_mape(
    y_test,
    baseline_pred
)

print("Baseline Performance")
print("--------------------")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")
print(f"R²   : {baseline_r2:.4f}")
print(f"MAPE : {baseline_mape:.2f}%")

In [0]:
comparison = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R2",
        "MAPE"
    ],

    "Baseline": [
        baseline_mae,
        baseline_rmse,
        baseline_r2,
        baseline_mape
    ],

    "LightGBM": [
        mae,
        rmse,
        r2,
        mape
    ]
})

display(comparison)

In [0]:
mae_improvement = (
    (baseline_mae - mae)
    / baseline_mae
) * 100

rmse_improvement = (
    (baseline_rmse - rmse)
    / baseline_rmse
) * 100

mape_improvement = (
    (baseline_mape - mape)
    / baseline_mape
) * 100

print(f"MAE improvement  : {mae_improvement:.2f}%")

print(f"RMSE improvement : {rmse_improvement:.2f}%")

print(f"MAPE improvement : {mape_improvement:.2f}%")

In [0]:
plt.figure(figsize=(14, 6))

plt.plot(test_df["Date"].values[:200],y_test.values[:200],label="Actual")

plt.plot(test_df["Date"].values[:200],y_pred[:200],label="Predicted")

plt.xlabel("Date")
plt.ylabel("Demand")
plt.title("Actual vs Predicted Demand")

plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8, 8))

plt.scatter(y_test,y_pred, alpha=0.5)

min_value = min(y_test.min(), y_pred.min())

max_value = max(y_test.max(), y_pred.max())

plt.plot([min_value, max_value], [min_value, max_value],linestyle="--")

plt.xlabel("Actual Demand")
plt.ylabel("Predicted Demand")

plt.title("Actual vs Predicted Demand")

plt.tight_layout()
plt.show()

In [0]:
# Residual = Actual - Predicted

residuals = (y_test.values - y_pred)

evaluation_df = test_df[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Category",
        "Region",
        "Demand"
    ]
].copy()

evaluation_df["Predicted"] = y_pred
evaluation_df["Residual"] = residuals

display(evaluation_df.head(20))

In [0]:
plt.figure(figsize=(10, 6))

plt.hist(residuals,bins=50)

plt.xlabel("Residual")
plt.ylabel("Frequency")

plt.title("Residual Distribution")

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(10, 6))

plt.scatter(y_pred,residuals,alpha=0.5)

plt.axhline(y=0,linestyle="--")

plt.xlabel("Predicted Demand")
plt.ylabel("Residual")

plt.title("Residual vs Predicted Demand")

plt.tight_layout()
plt.show()

In [0]:
category_evaluation = (
    evaluation_df
    .assign(
        Absolute_Error=lambda x:
        np.abs(
            x["Demand"] - x["Predicted"]
        )
    )
    .groupby("Category")
    .agg(
        Actual_Demand=("Demand", "mean"),
        Predicted_Demand=("Predicted", "mean"),
        MAE=("Absolute_Error", "mean"),
        Count=("Demand", "count")
    )
    .sort_values(
        "MAE",
        ascending=False
    )
)

display(category_evaluation)

In [0]:
region_evaluation = (
    evaluation_df
    .assign(
        Absolute_Error=lambda x:
        np.abs(
            x["Demand"] - x["Predicted"]
        )
    )
    .groupby("Region")
    .agg(
        Actual_Demand=("Demand", "mean"),
        Predicted_Demand=("Predicted", "mean"),
        MAE=("Absolute_Error", "mean"),
        Count=("Demand", "count")
    )
    .sort_values(
        "MAE",
        ascending=False
    )
)

display(region_evaluation)

In [0]:
worst_predictions = (
    evaluation_df
    .assign(
        Absolute_Error=lambda x:
        np.abs(
            x["Demand"] - x["Predicted"]
        )
    )
    .sort_values(
        "Absolute_Error",
        ascending=False
    )
)

display(
    worst_predictions.head(20)
)

In [0]:
best_predictions = (
    evaluation_df
    .assign(
        Absolute_Error=lambda x:
        np.abs(
            x["Demand"] - x["Predicted"]
        )
    )
    .sort_values(
        "Absolute_Error",
        ascending=True
    )
)

display(
    best_predictions.head(20)
)

In [0]:
with mlflow.start_run(
    run_id=run_id
):

    mlflow.log_metrics({
        "test_mae": mae,
        "test_rmse": rmse,
        "test_r2": r2,
        "test_mape": mape,

        "baseline_mae": baseline_mae,
        "baseline_rmse": baseline_rmse,
        "baseline_r2": baseline_r2,
        "baseline_mape": baseline_mape,

        "mae_improvement_pct": mae_improvement,
        "rmse_improvement_pct": rmse_improvement,
        "mape_improvement_pct": mape_improvement
    })

print("Evaluation metrics logged to MLflow.")

In [0]:
print("=" * 60)
print("DEMAND FORECASTING - FINAL EVALUATION")
print("=" * 60)

print("\nLightGBM")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

print("\nBaseline")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")
print(f"R²   : {baseline_r2:.4f}")
print(f"MAPE : {baseline_mape:.2f}%")

print("\nImprovement")
print(f"MAE  : {mae_improvement:.2f}%")
print(f"RMSE : {rmse_improvement:.2f}%")
print(f"MAPE : {mape_improvement:.2f}%")

print("=" * 60)